# Phase C 

In [3]:
import pandas as pd

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (classification_report,precision_score,recall_score,f1_score)

sms = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"]
)

display(sms.head())

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Répartition des classes

In [4]:
print("Nombre de messages par classe :")
print(sms["label"].value_counts())

print("\nPourcentage de messages par classe :")
print(
    (sms["label"].value_counts(normalize=True) * 100)
    .round(2)
)

Nombre de messages par classe :
label
ham     4825
spam     747
Name: count, dtype: int64

Pourcentage de messages par classe :
label
ham     86.59
spam    13.41
Name: proportion, dtype: float64


### Préparation des données

In [5]:
X_textes = sms["message"]

y = sms["label"].map({
    "ham": 0,
    "spam": 1
})

print("Nombre de textes :", X_textes.shape[0])
print("Nombre d'étiquettes :", y.shape[0])

print("\nCorrespondance utilisée :")
print("0 = normal")
print("1 = spam")

Nombre de textes : 5572
Nombre d'étiquettes : 5572

Correspondance utilisée :
0 = normal
1 = spam


### Découpage entraînement/test

In [6]:
X_train_textes, X_test_textes, y_train, y_test = train_test_split(
    X_textes,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Messages d'entraînement :", X_train_textes.shape[0])
print("Messages de test :", X_test_textes.shape[0])

print("\nRépartition dans le jeu de test :")
print(y_test.value_counts())

Messages d'entraînement : 4457
Messages de test : 1115

Répartition dans le jeu de test :
label
0    966
1    149
Name: count, dtype: int64


### Fonction vectoriser_textes

In [8]:
def vectoriser_textes(messages, vectorizer=None):
    """
    Transforme une liste de textes en matrice numérique.

    Si vectorizer est None, un nouveau TfidfVectorizer est créé
    et entraîné avec fit_transform.

    Sinon, le vectorizer existant est réutilisé avec transform.

    Retourne la matrice numérique et le vectorizer.
    """

    if vectorizer is None:
        vectorizer = TfidfVectorizer()
        X_vectorise = vectorizer.fit_transform(messages)
    else:
        X_vectorise = vectorizer.transform(messages)

    return X_vectorise, vectorizer
X_train, vectorizer = vectoriser_textes(
    X_train_textes
)

print("Dimensions de X_train :", X_train.shape)
print("Nombre de mots appris :", len(vectorizer.vocabulary_))
X_test, _ = vectoriser_textes(
    X_test_textes,
    vectorizer
)

print("Dimensions de X_test :", X_test.shape)

Dimensions de X_train : (4457, 7668)
Nombre de mots appris : 7668
Dimensions de X_test : (1115, 7668)


### Fonction evaluer_spam

In [9]:
def evaluer_spam(modele, X_train, X_test, y_train, y_test):
    """
    Entraîne le modèle, effectue les prédictions
    et affiche le rapport de classification.

    Retourne la précision, le rappel et le score F1
    de la classe spam.
    """

    modele.fit(X_train, y_train)

    predictions = modele.predict(X_test)

    print(
        classification_report(
            y_test,
            predictions,
            labels=[0, 1],
            target_names=["normal", "spam"],
            zero_division=0
        )
    )

    precision = precision_score(
        y_test,
        predictions,
        pos_label=1,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        pos_label=1,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        pos_label=1,
        zero_division=0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


#### cas normal 

#### Appel avec Naive Bayes

In [31]:
modele_naive_bayes = MultinomialNB(
    alpha=0.1
)
print("Multinomial Naive Bayes ajusté \n")

resultats_naive_bayes = evaluer_spam(
    modele_naive_bayes,
    X_train,
    X_test,
    y_train,
    y_test
)




Multinomial Naive Bayes ajusté 

              precision    recall  f1-score   support

      normal       0.98      1.00      0.99       966
        spam       1.00      0.89      0.94       149

    accuracy                           0.98      1115
   macro avg       0.99      0.94      0.97      1115
weighted avg       0.99      0.98      0.98      1115



#### Appel avec la régression logistique

In [33]:
modele_logistique = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

print("Régression logistique équilibrée \n")

resultats_logistique = evaluer_spam(
    modele_logistique,
    X_train,
    X_test,
    y_train,
    y_test
)


Régression logistique équilibrée 

              precision    recall  f1-score   support

      normal       0.99      0.99      0.99       966
        spam       0.95      0.93      0.94       149

    accuracy                           0.98      1115
   macro avg       0.97      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115



#### Comparaison des modèle

In [34]:
comparaison_modeles = pd.DataFrame({
    "Naive Bayes": resultats_naive_bayes,
    "Régression logistique": resultats_logistique
}).T

comparaison_modeles = comparaison_modeles.sort_values(
    by="f1",
    ascending=False
)

print("Comparaison des modèles ajustés :")

display(comparaison_modeles.round(3))

Comparaison des modèles ajustés :


,precision,recall,f1
Naive Bayes,1.000,0.886,0.940
Régression logistique,0.952,0.926,0.939


#### Happy path

In [35]:
print("Vérification du Happy path :\n")

for nom_modele, resultats in {
    "Naive Bayes": resultats_naive_bayes,
    "Régression logistique": resultats_logistique
}.items():

    rappel = resultats["recall"]

    print(f"{nom_modele} : recall spam = {rappel:.3f}")

    if rappel > 0.85:
        print("Happy path validé : la majorité des spams est détectée.\n")
    else:
        print("Happy path non validé : rappel inférieur à 0,85.\n")

Vérification du Happy path :

Naive Bayes : recall spam = 0.886
Happy path validé : la majorité des spams est détectée.

Régression logistique : recall spam = 0.926
Happy path validé : la majorité des spams est détectée.



#### Edge case

##### vectorisation du message vide

In [36]:
message_vide = [""]

X_message_vide, _ = vectoriser_textes(
    message_vide,
    vectorizer
)

print("Dimensions du message vide vectorisé :")
print(X_message_vide.shape)

print("\nNombre de valeurs non nulles :")
print(X_message_vide.nnz)

Dimensions du message vide vectorisé :
(1, 7668)

Nombre de valeurs non nulles :
0


#### prédiction de Naive Bayes

In [37]:
prediction_vide_nb = modele_naive_bayes.predict(
    X_message_vide
)[0]

print("Naive Bayes prédit :", prediction_vide_nb)

Naive Bayes prédit : 0


#### prédiction de la régression logistique

In [38]:
prediction_vide_logistique = modele_logistique.predict(
    X_message_vide
)[0]

print(
    "Régression logistique prédit :",
    prediction_vide_logistique
)

Régression logistique prédit : 0


#### Adversarial

#### vectorisation du spam déguisé

In [39]:
spam_deguise = [
    "salut, ton colis t attend, confirme ici"
]

X_spam_deguise, _ = vectoriser_textes(
    spam_deguise,
    vectorizer
)

print("Message testé :")
print(spam_deguise[0])

print("\nNombre de mots reconnus par le vectorizer :")
print(X_spam_deguise.nnz)

Message testé :
salut, ton colis t attend, confirme ici

Nombre de mots reconnus par le vectorizer :
1


#### résultat de Naive Bayes

In [40]:
prediction_spam_nb = modele_naive_bayes.predict(
    X_spam_deguise
)[0]

print("Naive Bayes prédit :", prediction_spam_nb)

if prediction_spam_nb == "ham":
    print(
        "Naive Bayes se fait tromper : "
        "le message est classé comme normal."
    )
else:
    print("Naive Bayes détecte le message comme spam.")

Naive Bayes prédit : 0
Naive Bayes détecte le message comme spam.


#### résultat de la régression logistique

In [41]:
prediction_spam_logistique = modele_logistique.predict(
    X_spam_deguise
)[0]

print(
    "Régression logistique prédit :",
    prediction_spam_logistique
)

if prediction_spam_logistique == "ham":
    print(
        "La régression logistique se fait tromper : "
        "le message est classé comme normal."
    )
else:
    print(
        "La régression logistique détecte "
        "le message comme spam."
    )

Régression logistique prédit : 0
La régression logistique détecte le message comme spam.


#### précision et rappel

In [42]:
print("Naive Bayes :")

print(
    "Précision spam :",
    round(resultats_naive_bayes["precision"], 3)
)

print(
    "Rappel spam :",
    round(resultats_naive_bayes["recall"], 3)
)

print("\nRégression logistique :")

print(
    "Précision spam :",
    round(resultats_logistique["precision"], 3)
)

print(
    "Rappel spam :",
    round(resultats_logistique["recall"], 3)
)

Naive Bayes :
Précision spam : 1.0
Rappel spam : 0.886

Régression logistique :
Précision spam : 0.952
Rappel spam : 0.926
